# 06B — Model B Cross-Period Validation

**Purpose:** Validate the *existing* cross-period forecasting outputs from `06_model_b_cross_period.ipynb` without retraining models. Checks include source data alignment, strict historical/recent/test windows, calendar-day forecasting, prediction file integrity, recomputed MAE/RMSE/R², and reasonable evidence against leakage.

**Caution:** Passing automated checks is not a mathematical proof of absence of leakage. Verify the training pipeline and feature-construction code in notebook 06 as well. The date and target comparisons are especially important.

**Run order:** Run notebook 06 to generate all CSVs, then run this validation notebook top to bottom. It will not modify your model notebook.


## 1. Imports and project paths

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'Cleaned'
RESULTS = ROOT / 'reports' / 'results' / 'modelb'
RESULTS.mkdir(parents=True, exist_ok=True)

OLD_CSV = DATA / 'city_day_clean.csv'
NEW_CSV = DATA / 'aqi_2023_2025_clean.csv'
PRED_CSV = RESULTS / 'predictions_same_test_rows.csv'
ALL_RESULTS_CSV = RESULTS / 'all_models_same_test_results.csv'
HIST_RESULTS_CSV = RESULTS / 'historical_model_results.csv'

RECENT_TRAIN_START = pd.Timestamp('2023-01-01')
HISTORICAL_END = pd.Timestamp('2020-12-31')
RECENT_TEST_START = pd.Timestamp('2024-11-01')
RECENT_TEST_END = pd.Timestamp('2025-04-30')

required_files = [OLD_CSV, NEW_CSV, PRED_CSV, ALL_RESULTS_CSV, HIST_RESULTS_CSV]
for path in required_files:
    print(('FOUND  ' if path.is_file() else 'MISSING'), path)
if any(not p.is_file() for p in required_files):
    raise FileNotFoundError('Run 06_model_b_cross_period.ipynb first and verify its dataset paths.')


FOUND   D:\IISER\7th sem\idc409(data science)\AQI_project_idc409\AQI-prediction-India\data\Cleaned\city_day_clean.csv
FOUND   D:\IISER\7th sem\idc409(data science)\AQI_project_idc409\AQI-prediction-India\data\Cleaned\aqi_2023_2025_clean.csv
FOUND   D:\IISER\7th sem\idc409(data science)\AQI_project_idc409\AQI-prediction-India\reports\results\modelb\predictions_same_test_rows.csv
FOUND   D:\IISER\7th sem\idc409(data science)\AQI_project_idc409\AQI-prediction-India\reports\results\modelb\all_models_same_test_results.csv
FOUND   D:\IISER\7th sem\idc409(data science)\AQI_project_idc409\AQI-prediction-India\reports\results\modelb\historical_model_results.csv


## 2. Read, standardize and audit source datasets

Reconstruct the exact City/Date/AQI cleaning logic used in notebook 06. Out-of-range AQI and duplicate city-date handling are made explicit.

In [2]:
CITY_ALIASES = {'Gurgaon':'Gurugram', 'New Delhi':'Delhi', 'Bangalore':'Bengaluru'}

def normalize(raw, source):
    rename = {'area':'City', 'city':'City', 'date':'Date', 'aqi_value':'AQI', 'aqi':'AQI'}
    d = raw.rename(columns={k:v for k,v in rename.items() if k in raw and v not in raw}).copy()
    if not {'City','Date','AQI'}.issubset(d.columns):
        raise ValueError(f'{source}: missing City, Date or AQI')
    d = d[['City','Date','AQI']].copy()
    iso = pd.to_datetime(d['Date'],format='%Y-%m-%d',errors='coerce')
    dmy = pd.to_datetime(d['Date'],format='%d-%m-%Y',errors='coerce')
    d['Date'] = (iso if iso.notna().sum() >= dmy.notna().sum() else dmy).dt.normalize()
    d['City'] = d['City'].astype('string').str.strip().str.title().replace(CITY_ALIASES)
    d['AQI'] = pd.to_numeric(d['AQI'],errors='coerce')
    total = len(d)
    d = d.dropna(subset=['City','Date','AQI'])
    bad_range = (~d['AQI'].between(0,500)).sum()
    d = d[d['AQI'].between(0,500)]
    dup = d.duplicated(['City','Date'],keep=False).sum()
    d = d.groupby(['City','Date'],as_index=False)['AQI'].median()
    print(f'{source}: raw={total:,}, invalid AQI range={bad_range:,}, duplicate rows={dup:,}, cleaned={len(d):,}')
    return d.sort_values(['City','Date']).reset_index(drop=True)

old = normalize(pd.read_csv(OLD_CSV,low_memory=False),'Historical')
new = normalize(pd.read_csv(NEW_CSV,low_memory=False),'Recent')
old = old.loc[old.Date <= HISTORICAL_END].copy()
new = new.loc[new.Date >= RECENT_TRAIN_START].copy()
common_cities = set(old.City) & set(new.City)
old = old[old.City.isin(common_cities)].copy()
new = new[new.City.isin(common_cities)].copy()
print('Common cities:', len(common_cities))
if not common_cities:
    raise ValueError('No common cities. Inspect your source datasets / alias mappings.')


Historical: raw=24,801, invalid AQI range=543, duplicate rows=0, cleaned=24,258
Recent: raw=235,005, invalid AQI range=0, duplicate rows=0, cleaned=235,005
Common cities: 25


## 3. Recreate temporal rows and evaluate split boundaries

This uses the same calendar reindexing, lag windows and dropna rules as notebook 06. It does not load future AQI into predictor features.

In [3]:
NUMERIC = ['AQI','AQI_lag1','AQI_lag2','AQI_lag3','AQI_lag7',
           'AQI_roll3','AQI_roll7','AQI_roll14','AQI_std7','AQI_change1',
           'month','weekday','is_weekend','is_winter']
CATEGORICAL = ['City','season']
FEATURES = NUMERIC + CATEGORICAL

def make_aqi_features(d):
    parts = []
    for city,g in d.groupby('City',sort=True):
        g = g.sort_values('Date').set_index('Date')
        daily = g[['AQI']].reindex(pd.date_range(g.index.min(),g.index.max(),freq='D'))
        daily.index.name = 'Date'
        daily['City'] = city
        for lag in (1,2,3,7):
            daily[f'AQI_lag{lag}'] = daily['AQI'].shift(lag)
        for w in (3,7,14):
            daily[f'AQI_roll{w}'] = daily['AQI'].shift(1).rolling(w,min_periods=max(2,int(w*0.7))).mean()
        daily['AQI_std7'] = daily['AQI'].shift(1).rolling(7,min_periods=5).std()
        daily['AQI_change1'] = daily['AQI'] - daily['AQI_lag1']
        daily['target_AQI_next'] = daily['AQI'].shift(-1)
        daily['target_date'] = daily.index + pd.Timedelta(days=1)
        daily['month'] = daily.index.month
        daily['weekday'] = daily.index.dayofweek
        daily['is_weekend'] = (daily['weekday'] >= 5).astype(int)
        daily['is_winter'] = daily['month'].isin([11,12,1,2]).astype(int)
        daily['season'] = np.select(
            [daily['month'].isin([12,1,2]),daily['month'].isin([3,4,5]),daily['month'].isin([6,7,8,9])],
            ['Winter','Summer','Monsoon'],default='Post-Monsoon')
        parts.append(daily.reset_index())
    if not parts:
        raise ValueError('No feature rows created')
    feat = pd.concat(parts,ignore_index=True).dropna(subset=NUMERIC + ['target_AQI_next'])
    return feat.sort_values(['Date','City']).reset_index(drop=True)

hf = make_aqi_features(old)
rf = make_aqi_features(new)
historical_train = hf[hf.target_date <= HISTORICAL_END].copy()
recent_train = rf[rf.target_date < RECENT_TEST_START].copy()
test = rf[rf.target_date.between(RECENT_TEST_START, RECENT_TEST_END)].copy()
valid_cities = set(historical_train.City) & set(recent_train.City) & set(test.City)
historical_train = historical_train[historical_train.City.isin(valid_cities)].copy()
recent_train = recent_train[recent_train.City.isin(valid_cities)].copy()
test = test[test.City.isin(valid_cities)].copy()
for label,d in [('Historical train',historical_train),('Recent train',recent_train),('Test',test)]:
    print(f'{label}: {len(d):,} rows, {d.City.nunique()} cities, targets {d.target_date.min()} through {d.target_date.max()}')


Historical train: 20,112 rows, 23 cities, targets 2015-01-11 00:00:00 through 2020-07-01 00:00:00
Recent train: 12,753 rows, 23 cities, targets 2023-01-11 00:00:00 through 2024-10-31 00:00:00
Test: 3,604 rows, 23 cities, targets 2024-11-01 00:00:00 through 2025-04-30 00:00:00


## 4. Run temporal and source-alignment checks

`PASS` denotes only the stated property, not comprehensive validation of the full modelling code.

In [4]:
checks = []
def check(name, condition, detail=''):
    checks.append({'Check':name,'Status':'PASS' if bool(condition) else 'FAIL','Detail':detail})

check('All splits nonempty', min(map(len,[historical_train,recent_train,test])) > 0)
if all(len(x)>0 for x in [historical_train,recent_train,test]):
    check('Historical train ends by 2020-12-31', historical_train.target_date.max() <= HISTORICAL_END)
    check('Historical train strictly precedes recent training', historical_train.target_date.max() < recent_train.target_date.min())
    check('Recent training precedes test', recent_train.target_date.max() < test.target_date.min())
    check('Test within requested interval', test.target_date.between(RECENT_TEST_START,RECENT_TEST_END).all())

for name,d in [('historical train',historical_train),('recent train',recent_train),('test',test)]:
    check(f'{name}: no duplicated City-Date', not d.duplicated(['City','Date']).any())
    check(f'{name}: target is next calendar day', (d.target_date - d.Date).eq(pd.Timedelta(days=1)).all())
    check(f'{name}: predictors and target nonmissing',d[FEATURES+['target_AQI_next']].notna().all().all())

# Exhaustive alignment against the *independently standardized source*.
for label, source, frame in [('historical',old,historical_train),('recent train',new,recent_train),('test',new,test)]:
    lookup = source[['City','Date','AQI']].rename(columns={'Date':'target_date','AQI':'actual_next'})
    merged = frame[['City','Date','target_date','target_AQI_next']].merge(lookup,on=['City','target_date'],how='left',validate='many_to_one')
    ok = merged.actual_next.notna().all() and np.allclose(merged.target_AQI_next,merged.actual_next,rtol=0,atol=1e-8)
    check(f'{label}: targets match actual AQI on next date',ok,f'rows checked: {len(merged)}')
    for lag in [1,2,3,7]:
        past = source[['City','Date','AQI']].copy()
        past['Date'] = past['Date'] + pd.Timedelta(days=lag)
        past = past.rename(columns={'AQI':'source_lag'})
        merged_lag = frame[['City','Date',f'AQI_lag{lag}']].merge(past,on=['City','Date'],how='left',validate='many_to_one')
        lag_ok = merged_lag.source_lag.notna().all() and np.allclose(merged_lag[f'AQI_lag{lag}'],merged_lag.source_lag,rtol=0,atol=1e-8)
        check(f'{label}: lag {lag} uses actual calendar day',lag_ok)

display(pd.DataFrame(checks))


,Check,Status,Detail
0,All splits nonempty,PASS,
1,Historical train ends by 2020-12-31,PASS,
2,Historical train strictly precedes recent trai...,PASS,
3,Recent training precedes test,PASS,
4,Test within requested interval,PASS,
5,historical train: no duplicated City-Date,PASS,
6,historical train: target is next calendar day,PASS,
7,historical train: predictors and target nonmis...,PASS,
8,recent train: no duplicated City-Date,PASS,
9,recent train: target is next calendar day,PASS,


## 5. Read prediction files and verify one-to-one test matching

The saved prediction rows must match precisely the reconstructed held-out test rows (City, Date, target date and actual AQI).

In [5]:
pred = pd.read_csv(PRED_CSV,parse_dates=['Date','target_date'])
all_results = pd.read_csv(ALL_RESULTS_CSV)
hist_results = pd.read_csv(HIST_RESULTS_CSV)
required = {'City','Date','target_date','actual','Persistence'}
missing = required - set(pred.columns)
if missing:
    raise ValueError(f'Prediction CSV missing columns: {missing}')
check('Prediction count equals reconstructed test',len(pred)==len(test),f'{len(pred)} saved vs {len(test)} reconstructed')
check('Prediction keys unique',not pred.duplicated(['City','Date']).any())
match = pred.merge(test[['City','Date','target_date','target_AQI_next','AQI']],on=['City','Date'],how='outer',indicator=True,suffixes=('','_source'),validate='one_to_one')
check('Saved rows correspond to exact test keys',match['_merge'].eq('both').all())
matched = match[match['_merge'].eq('both')].copy()
check('Saved target dates match',matched.target_date.eq(matched.target_date_source).all())
check('Saved actual AQI values match',np.allclose(matched.actual,matched.target_AQI_next,rtol=0,atol=1e-8))
check('Persistence equals same-day AQI',np.allclose(matched.Persistence,matched.AQI,rtol=0,atol=1e-8))
print('Historical, recent and baseline model prediction columns:')
print([c for c in pred.columns if c.startswith(('Historical - ','Recent - ')) or c=='Persistence'])


Historical, recent and baseline model prediction columns:
['Persistence', 'Historical - Ridge Regression', 'Historical - Random Forest', 'Historical - Hist Gradient Boosting', 'Recent - Ridge Regression', 'Recent - Random Forest', 'Recent - Hist Gradient Boosting']


## 6. Recalculate MAE, RMSE and R² from exported predictions

In [6]:
recomputed=[]
model_cols=[c for c in pred.columns if c.startswith(('Historical - ','Recent - ')) or c=='Persistence']
for col in model_cols:
    y = pd.to_numeric(pred['actual'],errors='coerce')
    p = pd.to_numeric(pred[col],errors='coerce')
    finite = np.isfinite(y).all() and np.isfinite(p).all()
    check(f'{col}: all predictions finite',finite)
    if not finite:
        continue
    training = ('Historical 2015–2020' if col.startswith('Historical - ') else
                'Recent pre-test (2023–2024)' if col.startswith('Recent - ') else 'No fitting')
    model = col.split(' - ',1)[1] if col != 'Persistence' else 'Persistence (tomorrow = today)'
    recomputed.append({'Training':training,'Model':model,'MAE':mean_absolute_error(y,p),
                       'RMSE':float(np.sqrt(mean_squared_error(y,p))),'R2':r2_score(y,p)})
recomputed=pd.DataFrame(recomputed)
display(recomputed.round(4))
comp = recomputed.merge(all_results,on=['Training','Model'],how='outer',suffixes=('_recalc','_saved'),indicator=True,validate='one_to_one')
check('Every saved metric row is present',comp['_merge'].eq('both').all())
for metric in ['MAE','RMSE','R2']:
    a=comp[f'{metric}_recalc']; b=comp[f'{metric}_saved']
    check(f'{metric} reproduces report',a.notna().all() and b.notna().all() and np.allclose(a,b,atol=1e-6,rtol=1e-6))

# Notebook 06 writes this before fitting recent models: it must contain only baseline/historical rows.
hist_keys=set(zip(hist_results['Training'],hist_results['Model']))
expected_hist_keys=set(zip(recomputed.loc[recomputed.Training!='Recent pre-test (2023–2024)','Training'],
                          recomputed.loc[recomputed.Training!='Recent pre-test (2023–2024)','Model']))
check('Historical results CSV contains expected models',hist_keys==expected_hist_keys)


,Training,Model,MAE,RMSE,R2
0,No fitting,Persistence (tomorrow = today),24.2703,35.8260,0.7688
1,Historical 2015–2020,Ridge Regression,23.3362,33.4772,0.7982
2,Historical 2015–2020,Random Forest,23.4578,33.5689,0.7971
3,Historical 2015–2020,Hist Gradient Boosting,23.2103,33.3628,0.7995
4,Recent pre-test (2023–2024),Ridge Regression,23.0899,33.1189,0.8025
5,Recent pre-test (2023–2024),Random Forest,23.5316,33.9350,0.7926
6,Recent pre-test (2023–2024),Hist Gradient Boosting,23.3128,33.6604,0.7959


## 7. Final report and manual review

Automation cannot determine whether the model-training code was altered after results were generated. Before claiming validated performance, compare notebook 06 with the checked-in version and verify:

- Imputation, scaling and one-hot encoding are fitted inside the pipeline using **training only**.
- `make_aqi_features` uses current-day/older information for predictors; `target_AQI_next` is strictly next calendar day.
- Historical training fits on 2015–2020; recent training fits only before the test interval.
- No feature selection, tuning, or model choice uses the held-out test to repeatedly optimize a final model. Choosing the visually best test model is **descriptive**, not an unbiased selected-model estimate.
- Differences in training populations, missingness, and AQI distributions are reported as limitations.
